In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
data = [(1, 'Eva', 2450), (2, 'Charlie', 3097), (3, 'Ivy', 1953), (4, 'Bob', 3924), (5, 'Bob', 4746), (6, 'Frank', 3120), (7, 'Frank', 1431), (8, 'Frank', 801), (9, 'Jack', 4388), (10, 'Eva', 3268), (11, 'Ivy', 2794), (12, 'Jack', 4153), (13, 'Alice', 4410), (14, 'Ivy', 2865), (15, 'Frank', 2337), (16, 'Grace', 2575), (17, 'Charlie', 2446), (18, 'Bob', 2462), (19, 'Frank', 682), (20, 'Charlie', 819), (21, 'Eva', 4654), (22, 'Jack', 3449), (23, 'Ivy', 4630), (24, 'Eva', 3654), (25, 'Bob', 2662), (26, 'Frank', 741), (27, 'Ivy', 3688), (28, 'Jack', 4729), (29, 'Alice', 2930), (30, 'Charlie', 1085), (31, 'David', 4096), (32, 'Helen', 523), (33, 'Jack', 868), (34, 'Eva', 3094), (35, 'Charlie', 4403), (36, 'Bob', 1953), (37, 'Frank', 1996), (38, 'Alice', 1930), (39, 'Grace', 4408), (40, 'Alice', 4145), (41, 'David', 2478), (42, 'Charlie', 3403), (43, 'David', 2553), (44, 'Grace', 2456), (45, 'Charlie', 3523), (46, 'Frank', 2434), (47, 'Charlie', 2844), (48, 'Jack', 3830), (49, 'Frank', 1889), (50, 'Grace', 3858), (51, 'Eva', 4345), (52, 'Alice', 1584), (53, 'Charlie', 1359), (54, 'Bob', 3538), (55, 'Charlie', 983), (56, 'Frank', 696), (57, 'Alice', 675), (58, 'Alice', 3153), (59, 'Alice', 1802), (60, 'Eva', 4376)]

df = spark.createDataFrame(data, ["id","name","amount"])

In [0]:
df.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/practice/ass6")
df=spark.read.format("delta").load("/Volumes/workspace/default/practice/ass6")
df.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


In [0]:
incremental_data = [
    (1, "Alice", 3000),   # update
    (2, "Bob", 4000),     # update
    (61, "NewCust1", 2000),  # insert
    (62, "NewCust2", 3500)   # insert
]
inc_df = spark.createDataFrame(incremental_data, ["id","name","amount"])

In [0]:
inc_df.write.format("delta").mode("append").save("/Volumes/workspace/default/practice/ass6")
inc_df=spark.read.format("delta").load("/Volumes/workspace/default/practice/ass6")
inc_df.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


In [0]:
from delta.tables import *
de=DeltaTable.forPath(spark,"/Volumes/workspace/default/practice/ass6").delete("amount < 1000")


In [0]:
inc_df = inc_df.dropDuplicates(["id"])

In [0]:
de = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/practice/ass6"
)
de.alias("target").merge(
    inc_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(
    set={
        "name": "source.name",
        "amount": "source.amount"
    }
).whenNotMatchedInsert(
    values={
        "id": "source.id",
        "name": "source.name",
        "amount": "source.amount"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
de = [(90,"keerthi",7890,"good")]
new_col = ["id","name","amount","category"]

de_df = spark.createDataFrame(de,new_col)

de_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema","true") \
    .save("/Volumes/workspace/default/practice/ass6")


In [0]:
de_df=DeltaTable.forPath(spark,"/Volumes/workspace/default/practice/ass6")
de_df.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-04-14T16:21:53.000Z,73109523991213,keerthisrivalli18@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(458140601127140),b16bfb28-d89c-48cc-b0a8-21c297492d8a,0414-155727-ri483zvd-v2n,11,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1228)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-14T16:11:44.000Z,73109523991213,keerthisrivalli18@gmail.com,MERGE,"Map(predicate -> [""(id#12043L = id#11632L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(458140601127140),aa46c148-7396-4ba1-8172-3fa9e5e866c6,0414-155727-ri483zvd-v2n,10,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1635, numTargetBytesRemoved -> 1649, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 55, executionTimeMs -> 3741, materializeSourceTimeMs -> 638, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1476, numTargetRowsUpdated -> 55, numOutputRows -> 55, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 53, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1500)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-14T16:07:53.000Z,73109523991213,keerthisrivalli18@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(458140601127140),b4c096a6-94c1-4bd9-a823-71095a14d7d5,0414-155727-ri483zvd-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2779, p25FileSize -> 1649, numDeletionVectorsRemoved -> 1, minFileSize -> 1649, numAddedFiles -> 1, maxFileSize -> 1649, p75FileSize -> 1649, p50FileSize -> 1649, numAddedBytes -> 1649)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-14T16:07:50.000Z,73109523991213,keerthisrivalli18@gmail.com,DELETE,"Map(predicate -> [""(amount#11681L < 1000)""])",null,List(458140601127140),b4c096a6-94c1-4bd9-a823-71095a14d7d5,0414-155727-ri483zvd-v2n,8,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1754, numDeletionVectorsUpdated -> 0, numDeletedRows -> 9, scanTimeMs -> 1149, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 604)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-04-14T16:07:46.000Z,73109523991213,keerthisrivalli18@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(458140601127140),c1e1dee8-7f3e-4467-98fc-9e0590f8d460,0414-155727-ri483zvd-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1104)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-14T16:07:41.000Z,73109523991213,keerthisrivalli18@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(458140601127140),4edbdb81-55f8-4f9e-b82b-5f17ec1b9ce1,0414-155727-ri483zvd-v2n,6,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1641, numDeletionVectorsRemoved -> 0, numOutputRows -> 60, numOutputBytes -> 1675)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-14T16:05:12.000Z,73109523991213,keerthisrivalli18@gmail.com,DELETE,"Map(predicate -> [""(amount#11361L < 1000)""])",null,List(458140601127140),2801936e-af76-427d-afe3-2c402b7756a4,0414-155727-ri483zvd-v2n,5,WriteSerializable,false,"Map(numRemovedFiles

In [0]:
de_df=spark.read.format("delta").option("versionAsOf",0).load("/Volumes/workspace/default/practice/ass6")
de_df.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268
